# Get example datasets from CELLxGENE (non immune datasets)

In [1]:
import os
from os.path import join

import anndata
import pandas as pd
import tqdm

## Datasets to download

In [2]:
from dataclasses import dataclass
from typing import List


@dataclass
class CxGCollection:
    collection_id: str
    dataset_ids: List[str]
    celltype_cols: List[str]
    cell_type_author: str  # this is the most fine-grained annotation provided by the author
    cts: List[str]  # cell types labels to subset to
    sample_id: str  # this sequencing sample ID
    tissue: str

In [3]:
cxg_collections = [
    CxGCollection(
        collection_id="Liver_macrophages",
        dataset_ids=["bb930137-be57-42f1-9a86-dc69370404e8_updated"],
        celltype_cols=["author_cell_type"],
        cell_type_author="author_cell_type",
        cts=[
            "Activated",
            "Kupffer",
            "LAM-like",
            "MHCII/DC",
            "Resident",
            "Synapse",
            "Mono-Act",
            "Monocyte"
        ],
        sample_id="sample",
        tissue="caudate lobe of liver"
    ),
    CxGCollection(
        collection_id="Ileum_macrophages",
        dataset_ids=["25cc19dd-81eb-4e79-8820-86ff4cfb88b1_updated"],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        cts=[
            "Macrophages",
            "Macrophages LYVE1",
            "Macrophages CCL3 CCL4",
            "Macrophages CXCL9 CXCL10",
            "Macrophages PLA2G2D"
        ],
        sample_id="biosample_id",
        tissue="ileum"
    ),
    CxGCollection(
        collection_id="Colon_macrophages",
        dataset_ids=["f6f5913c-467b-40fe-800c-cd65375840ff_updated"],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        cts=[
            "Macrophages",
            "Macrophages LYVE1",
            "Macrophages CCL3 CCL4",
            "Macrophages Metallothionein"
        ],
        sample_id="biosample_id",
        tissue="colon"
    ),
    CxGCollection(
        collection_id="5c868b6f-62c5-4532-9d7f-a346ad4b50a7_Ileum_updated",
        dataset_ids=["25cc19dd-81eb-4e79-8820-86ff4cfb88b1_updated"],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        cts=None,
        sample_id="biosample_id",
        tissue="ileum"
    ),
    CxGCollection(
        collection_id="5c868b6f-62c5-4532-9d7f-a346ad4b50a7_Colon_updated",
        dataset_ids=["f6f5913c-467b-40fe-800c-cd65375840ff_updated"],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        cts=None,
        sample_id="biosample_id",
        tissue="colon"
    ),
    CxGCollection(
        collection_id="Colon_macrophages_felix",
        dataset_ids=["Colon_macrophages_felix"],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        cts=None,
        sample_id="biosample_id",
        tissue="colon"
    ),
    CxGCollection(
        collection_id="5c868b6f-62c5-4532-9d7f-a346ad4b50a7_Colon_updated_felix",
        dataset_ids=["f6f5913c-467b-40fe-800c-cd65375840ff_updated_felix"],
        celltype_cols=["Celltype"],
        cell_type_author="Celltype",
        cts=None,
        sample_id="biosample_id",
        tissue="colon"
    ),
]


## Download raw datasets

In [4]:
# DOWNLOAD_PATH = "/mnt/dssfs02/dataset-similarity/raw"
DOWNLOAD_PATH = "/vol/data/dataset-similarity/raw"

In [5]:
for collection in tqdm.tqdm(cxg_collections):
    for dataset in collection.dataset_ids:
        save_path = join(DOWNLOAD_PATH, f"{dataset}.h5ad")
        if not os.path.isfile(save_path):
            os.system(
                f"wget -q -P {DOWNLOAD_PATH} https://datasets.cellxgene.cziscience.com/{dataset}.h5ad"
            )
        else:
            print(f"File {dataset} already exists. Skipping...")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 13875.30it/s]

File bb930137-be57-42f1-9a86-dc69370404e8_updated already exists. Skipping...
File 25cc19dd-81eb-4e79-8820-86ff4cfb88b1_updated already exists. Skipping...
File f6f5913c-467b-40fe-800c-cd65375840ff_updated already exists. Skipping...
File 25cc19dd-81eb-4e79-8820-86ff4cfb88b1_updated already exists. Skipping...
File f6f5913c-467b-40fe-800c-cd65375840ff_updated already exists. Skipping...
File Colon_macrophages_felix already exists. Skipping...
File f6f5913c-467b-40fe-800c-cd65375840ff_updated_felix already exists. Skipping...


## Preprocess datasets

In [6]:
import h5py
import numpy as np
from anndata.experimental import read_elem
from pandas.api.types import is_string_dtype
from scipy.sparse import csc_matrix, csr_matrix

In [7]:
# SAVE_PATH = "/mnt/dssfs02/dataset-similarity/preprocessed"
SAVE_PATH = "/vol/data/dataset-similarity/preprocessed"

In [8]:
def streamline_count_matrix(x_raw, gene_names_raw, gene_names_ref):
    assert len(gene_names_raw) == len(set(gene_names_raw))
    assert len(gene_names_ref) == len(set(gene_names_ref))
    assert len(gene_names_raw) == x_raw.shape[1]
    assert np.isin(gene_names_raw, gene_names_ref).sum() == x_raw.shape[1]
    # For fast column-wise slicing matrix has to be in csc format
    assert isinstance(x_raw, csc_matrix)
    gene_names_raw, gene_names_ref = np.array(gene_names_raw), np.array(gene_names_ref)
    row, col = np.empty(x_raw.nnz, dtype='i8'), np.empty(x_raw.nnz, dtype='i8')
    data = np.empty(x_raw.nnz, dtype='f4')

    ctr = 0
    for i, gene in enumerate(gene_names_ref):
        if gene in gene_names_raw:
            gene_idx = np.where(gene == gene_names_raw)[0]
            assert gene_idx.size == 1
            gene_idx = gene_idx[0]
            x_col = x_raw[:, gene_idx]
            idxs_nnz = x_col.indices
            n_nnz = len(idxs_nnz)
            col[ctr:ctr+n_nnz] = i
            row[ctr:ctr+n_nnz] = idxs_nnz
            data[ctr:ctr+n_nnz] = x_col.data
            ctr += n_nnz

    return csr_matrix(
        (data, (row, col)),
        shape=(x_raw.shape[0], len(gene_names_ref)),
        dtype='f4'
    )


In [9]:
datasets = []
for collection in cxg_collections:
    for dataset in collection.dataset_ids:
        datasets.append(join(DOWNLOAD_PATH, f"{dataset}.h5ad"))

var_dfs = []
for dataset in datasets:
    with h5py.File(dataset) as f:
        var = read_elem(f["var"])[["feature_name"]]
        var_dfs.append(var)

var_concat = (
    pd.concat(var_dfs)
    .drop_duplicates()
    .set_index("feature_name")
    .sort_index()
)
# var_concat = var_concat[~var_concat.index.duplicated()]
# var_concat = var_concat[~var_concat.feature_id.duplicated()]
var_concat

""
feature_name
5S_rRNA
7SK
7SK_ENSG00000232512
7SK_ENSG00000254144
7SK_ENSG00000260682
...
hsa-mir-150
hsa-mir-335
hsa-mir-490


In [10]:
assert var_concat.index.is_unique
# assert var_concat.feature_id.is_unique

var_concat.to_parquet(join(SAVE_PATH, "genes-macrophages.parquet"))

In [11]:
from scipy.sparse import csr_matrix


# columns to keep for the preprocessed data
COLUMNS = [
    "assay", "cell_type", "development_stage", "disease", "donor_id", 
    "is_primary_data", "sex", "suspension_type", "tissue",
]


def preprocess_dataset(
    dataset_path: str, 
    var_set: pd.DataFrame, 
    columns: List[str], 
    cell_type_column: str,
    sample_id_column: str
):
    with h5py.File(dataset_path) as f:
        try:
            x = read_elem(f["raw"]["X"]).astype("f4").tocsc()
            var = read_elem(f["raw"]["var"])
            obs = read_elem(f["raw"]["obs"])
        except KeyError:
            # if raw doesn't exist -> use .X instead
            # according to CELLxGENE schema
            # https://github.com/chanzuckerberg/single-cell-curation/blob/main/schema/3.0.0/schema.md#x-matrix-layers
            x = read_elem(f["X"]).astype("f4").tocsc()
            var = read_elem(f["var"])
            obs = read_elem(f["obs"])

    # align feature spaces across datasets
    x = x[:, np.isin(var.feature_name.tolist(), var_set.index.tolist())]
    var = var[np.isin(var.feature_name.tolist(), var_set.index.tolist())]
    x = streamline_count_matrix(x, var.feature_name.tolist(), var_set.index.tolist())
    # subselect to desired obs columns
    obs = (
        obs[columns].copy()
        .assign(cell_type_author=lambda df: df[cell_type_column])
        .assign(sample_id=lambda df: df[sample_id_column])
    )
    # convert all columns with string dtype to categorical dtype
    for col in obs.columns:
        if is_string_dtype(obs[col]):
            obs[col] = obs[col].astype("category")

    return anndata.AnnData(X=x, obs=obs, var=var_set)


In [12]:
for collection in tqdm.tqdm(cxg_collections):
    save_path = join(SAVE_PATH, f"{collection.collection_id}.h5ad")
    if not os.path.isfile(save_path):
        adatas = []
        for dataset in collection.dataset_ids:
            columns = COLUMNS + collection.celltype_cols
            if collection.sample_id not in columns:
                columns.append(collection.sample_id)
            adata = preprocess_dataset(
                join(DOWNLOAD_PATH, f"{dataset}.h5ad"),
                var_concat,
                columns,
                collection.cell_type_author,
                collection.sample_id
            )
            adatas.append(adata)

        if len(adatas) > 1:
            adatas = anndata.concat(adatas)
            adatas.var = var_concat
        else:
            adatas = adatas[0]

        if collection.cts:
            adatas = adatas[adatas.obs["cell_type_author"].isin(
                collection.cts
            )]
            print(collection.collection_id, adatas.obs["cell_type_author"].unique().tolist())
        adatas.write(save_path, compression="gzip")


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:36<00:00,  5.17s/it]


## Create h5ad object for liver data

In [40]:
import anndata as ad

from os.path import join

In [41]:
PATH = "/vol/data/dataset-similarity/raw"

In [45]:
adata_liver = ad.read_h5ad(join(PATH, "bb930137-be57-42f1-9a86-dc69370404e8.h5ad"))
adata_liver = ad.AnnData(
    X=adata_liver.raw.X,
    var=adata_liver.var,
    obs=adata_liver.obs,
)
adata_liver

AnnData object with n_obs × n_vars = 11127 × 33363
    obs: 'mapped_reference_assembly', 'mapped_reference_annotation', 'alignment_software', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'organism_ontology_term_id', 'sample_uuid', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_derivation_process', 'tissue_type', 'suspension_depleted_cell_types', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_dissociation_time', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_uuid', 'assay_ontology_term_id', 'library_starting_quantity', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'Phase', 'Coarse_clusters', 'sample', 'cell_type', 'assay', 'disease', 'organism', 'se

In [46]:
adata_liver.X.max()

4343.0

In [47]:
adata_liver.var["feature_name"] = adata_liver.var["feature_name"].str.split("_").str[0]

In [48]:
adata_liver = adata_liver[:, ~adata_liver.var.feature_name.duplicated()]

In [49]:
adata_liver

View of AnnData object with n_obs × n_vars = 11127 × 33342
    obs: 'mapped_reference_assembly', 'mapped_reference_annotation', 'alignment_software', 'donor_id', 'donor_age', 'self_reported_ethnicity_ontology_term_id', 'donor_cause_of_death', 'donor_living_at_sample_collection', 'organism_ontology_term_id', 'sample_uuid', 'sample_preservation_method', 'tissue_ontology_term_id', 'development_stage_ontology_term_id', 'sample_derivation_process', 'tissue_type', 'suspension_depleted_cell_types', 'suspension_derivation_process', 'suspension_dissociation_reagent', 'suspension_dissociation_time', 'suspension_uuid', 'suspension_type', 'tissue_handling_interval', 'library_uuid', 'assay_ontology_term_id', 'library_starting_quantity', 'sequencing_platform', 'is_primary_data', 'cell_type_ontology_term_id', 'author_cell_type', 'disease_ontology_term_id', 'sex_ontology_term_id', 'nCount_RNA', 'nFeature_RNA', 'percent.mt', 'Phase', 'Coarse_clusters', 'sample', 'cell_type', 'assay', 'disease', 'organi

In [50]:
adata_liver.write_h5ad(
    join(PATH, "bb930137-be57-42f1-9a86-dc69370404e8_updated.h5ad"),
    compression="gzip"
)

## Create h5ad objects for colon/ileum data

In [ ]:
# The colon/ileum datasets from CxG are missing important marker genes
# -> Take orginial data from the papers and use meta data provided by CxG

In [128]:
import anndata as ad
import pandas as pd

from os.path import join

#### Ileum 

In [129]:
PATH = "/vol/data/dataset-similarity/ileum"

In [130]:
adata_ileum = ad.AnnData(
    X=ad.read_mtx(join(PATH, "TI_IMM.scp.raw.mtx")).X.T,
    var=pd.read_csv(join(PATH, "TI_IMM.scp.features.tsv"), sep="\t", header=None, index_col=0, names=["feature_id", "feature_name"]),
    obs=pd.read_csv(join(PATH, "TI_IMM.scp.barcodes.tsv"), sep="\t", header=None, index_col=0)
)

In [131]:
adata_ileum

AnnData object with n_obs × n_vars = 201072 × 28923
    var: 'feature_name'

In [132]:
adata_ileum_cxg = ad.read_h5ad("/vol/data/dataset-similarity/raw/25cc19dd-81eb-4e79-8820-86ff4cfb88b1.h5ad")

In [133]:
assert adata_ileum.obs.index.equals(adata_ileum_cxg.obs.index)

adata_ileum.obs = adata_ileum_cxg.obs

In [134]:
adata_ileum.var

,feature_name
feature_id,
ENSG00000238009,RP11-34P13.7
ENSG00000237683,AL627309.1
ENSG00000228463,AP006222.2
ENSG00000237094,RP4-669L17.10
ENSG00000235373,RP11-206L10.3
...,...
ENSG00000166329,CCDC182
ENSG00000267157,CTB-54O9.9
ENSG00000231253,RP1-302D9.2


In [138]:
"CCL4" in adata_ileum.var.feature_name.values

True

In [139]:
adata_ileum.write_h5ad(
    "/vol/data/dataset-similarity/raw/25cc19dd-81eb-4e79-8820-86ff4cfb88b1_updated.h5ad",
    compression="gzip"
)

#### Colon

In [140]:
PATH = "/vol/data/dataset-similarity/colon"

In [141]:
adata_colon = ad.AnnData(
    X=ad.read_mtx(join(PATH, "CO_IMM.scp.raw.mtx")).X.T,
    var=pd.read_csv(join(PATH, "CO_IMM.scp.features.tsv"), sep="\t", header=None, index_col=0, names=["feature_id", "feature_name"]),
    obs=pd.read_csv(join(PATH, "CO_IMM.scp.barcodes.tsv"), sep="\t", header=None, index_col=0)
)

In [142]:
adata_colon

AnnData object with n_obs × n_vars = 152509 × 28663
    var: 'feature_name'

In [143]:
adata_colon_cxg = ad.read_h5ad("/vol/data/dataset-similarity/raw/f6f5913c-467b-40fe-800c-cd65375840ff.h5ad")

In [144]:
assert adata_colon.obs.index.equals(adata_colon_cxg.obs.index)

adata_colon.obs = adata_colon_cxg.obs

In [145]:
adata_colon.var

,feature_name
feature_id,
ENSG00000237683,AL627309.1
ENSG00000228463,AP006222.2
ENSG00000237094,RP4-669L17.10
ENSG00000235373,RP11-206L10.3
ENSG00000228327,RP11-206L10.2
...,...
ENSG00000237832,RP5-974N19.1
ENSG00000268287,CTB-60B18.18
ENSG00000254760,CTD-2616J11.3


In [146]:
"CCL4" in adata_colon.var.feature_name.values

True

In [147]:
adata_colon.write_h5ad(
    "/vol/data/dataset-similarity/raw/f6f5913c-467b-40fe-800c-cd65375840ff_updated.h5ad",
    compression="gzip"
)